In [1]:
import torch
import torch.nn as nn

text = "hello world"

In [2]:
itos = {}
stoi = {}
unsorted_char = set()
for i in text:
    unsorted_char.add(i)
n = len(unsorted_char)
chars = sorted(unsorted_char)
for i , char in enumerate(chars):
    itos[i] = char
    stoi[char] = i

starts = [0, 6]
sequence_length = 4

encoded = []
for char in text:
    encoded.append(stoi[char])

data = torch.tensor(encoded, dtype=torch.long)
x = []
for i in starts:
    tmp = torch.tensor(encoded[i:i+4], dtype= torch.long)
    x.append(tmp)
y = []
for i in starts:
    tmp = torch.tensor(encoded[i+1:i+5], dtype= torch.long)
    y.append(tmp)

x_batch = torch.stack(x, dim=0)
y_batch = torch.stack(y, dim=0)

print(x_batch)
print(y_batch)
print(x_batch.shape, y_batch.shape)


tensor([[3, 2, 4, 4],
        [7, 5, 6, 4]])
tensor([[2, 4, 4, 5],
        [5, 6, 4, 1]])
torch.Size([2, 4]) torch.Size([2, 4])


In [3]:
table = nn.Embedding(n,n)
pred = table(x_batch)
print(pred)
print(x_batch.shape, pred.shape, table.weight.shape)

print(torch.allclose(pred[0,2] , table.weight[x_batch[0,2]]))

tensor([[[-0.6508, -0.3115,  1.6341,  0.8104, -1.9684, -0.3191,  0.0426,
          -0.5702],
         [-0.8274, -1.2680,  2.2047, -1.0490, -0.5989,  0.8636, -0.3010,
          -0.8467],
         [ 0.5795, -0.4705,  0.7897,  2.1296, -1.8040,  0.5982,  0.6163,
           0.4409],
         [ 0.5795, -0.4705,  0.7897,  2.1296, -1.8040,  0.5982,  0.6163,
           0.4409]],

        [[-1.6689,  0.0534,  0.6299, -0.2613,  0.8283, -0.2366, -1.7158,
          -0.5533],
         [-1.2672,  0.9003, -2.0206, -0.6028, -0.3502,  0.5069, -0.8873,
           1.2086],
         [ 0.0575, -1.0434,  0.4231,  1.1180,  0.2560,  0.0931,  0.0221,
           0.5175],
         [ 0.5795, -0.4705,  0.7897,  2.1296, -1.8040,  0.5982,  0.6163,
           0.4409]]], grad_fn=<EmbeddingBackward0>)
torch.Size([2, 4]) torch.Size([2, 4, 8]) torch.Size([8, 8])
True


In [6]:
flat_logits = pred.reshape(-1,8)
print(flat_logits)
flat_target = y_batch.reshape(-1)
print(flat_target)

loss_api = nn.CrossEntropyLoss()
loss = loss_api(flat_logits,flat_target)
print(torch.allclose(flat_logits[4], pred[1, 0]))
print(flat_target[4] == y_batch[1, 0])
print(loss)

tensor([[-0.6508, -0.3115,  1.6341,  0.8104, -1.9684, -0.3191,  0.0426, -0.5702],
        [-0.8274, -1.2680,  2.2047, -1.0490, -0.5989,  0.8636, -0.3010, -0.8467],
        [ 0.5795, -0.4705,  0.7897,  2.1296, -1.8040,  0.5982,  0.6163,  0.4409],
        [ 0.5795, -0.4705,  0.7897,  2.1296, -1.8040,  0.5982,  0.6163,  0.4409],
        [-1.6689,  0.0534,  0.6299, -0.2613,  0.8283, -0.2366, -1.7158, -0.5533],
        [-1.2672,  0.9003, -2.0206, -0.6028, -0.3502,  0.5069, -0.8873,  1.2086],
        [ 0.0575, -1.0434,  0.4231,  1.1180,  0.2560,  0.0931,  0.0221,  0.5175],
        [ 0.5795, -0.4705,  0.7897,  2.1296, -1.8040,  0.5982,  0.6163,  0.4409]],
       grad_fn=<ViewBackward0>)
tensor([2, 4, 4, 5, 5, 6, 4, 1])
True
tensor(True)
tensor(2.7521, grad_fn=<NllLossBackward0>)
